# DA6401 Assignment 2 — W&B Experiments

## Setup — GPU check & install packages

In [1]:
import torch
print('GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
!pip install -q albumentations wandb gdown
print('Packages ready.')

GPU: True
Device: Tesla T4
Packages ready.


## Setup — W&B Login

In [2]:
import wandb
wandb.login()  # paste your API key from https://wandb.ai/authorize

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: agrawalritik2001 (agrawalritik2001-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Setup — Upload & extract project zip

In [5]:
import os
os.listdir('/kaggle/input/datasets/ritikagrawa1/dl-ass-with-data/train/')  # Returns a list of all files and folders

['multitask.py',
 'train.py',
 'inference.py',
 'losses',
 'checkpoints',
 'README.md',
 'models',
 'requirements.txt',
 'data']

In [7]:
import os

PROJECT_DIR = "/kaggle/input/datasets/ritikagrawa1/dl-ass2-checkpoints/checkpoints"

os.chdir(PROJECT_DIR)

print(f"Working directory: {os.getcwd()}")
print("Files:", os.listdir())

Working directory: /kaggle/input/datasets/ritikagrawa1/dl-ass2-checkpoints/checkpoints
Files: ['unet.pth', 'localizer.pth', 'classifier.pth', 'checkpoints.md']


## Setup — Common imports & helpers

In [5]:
import wandb
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

from models.classification import VGG11Classifier
from models.localization import VGG11Localizer
from models.segmentation import VGG11UNet
from data.pets_dataset import OxfordIIITPetDataset
from losses.iou_loss import IoULoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
TRIMAP_COLORS = np.array([[0,200,0],[200,0,0],[200,200,0]], dtype=np.uint8)

def denorm(t):
    return ((t.cpu() * STD + MEAN).clamp(0,1).permute(1,2,0).numpy()*255).astype(np.uint8)

def batch_iou(pred, tgt, eps=1e-6):
    pw, ph = pred[:,2].clamp(0), pred[:,3].clamp(0)
    px1, px2 = pred[:,0]-pw/2, pred[:,0]+pw/2
    py1, py2 = pred[:,1]-ph/2, pred[:,1]+ph/2
    tw, th = tgt[:,2], tgt[:,3]
    tx1, tx2 = tgt[:,0]-tw/2, tgt[:,0]+tw/2
    ty1, ty2 = tgt[:,1]-th/2, tgt[:,1]+th/2
    iw = (torch.min(px2,tx2)-torch.max(px1,tx1)).clamp(0)
    ih = (torch.min(py2,ty2)-torch.max(py1,ty1)).clamp(0)
    inter = iw*ih; union = pw*ph + tw*th - inter
    return (inter/(union+eps)).mean().item()

def dice_score(logits, masks, eps=1e-6):
    preds = logits.argmax(1)
    fp = (preds==0).float(); ft = (masks==0).float()
    return ((2*(fp*ft).sum()+eps)/(fp.sum()+ft.sum()+eps)).item()

def draw_boxes(img_np, gt_box, pred_box):
    img = Image.fromarray(img_np)
    d = ImageDraw.Draw(img)
    xc,yc,w,h = gt_box
    d.rectangle([xc-w/2,yc-h/2,xc+w/2,yc+h/2], outline='green', width=3)
    xc,yc,w,h = pred_box
    d.rectangle([xc-w/2,yc-h/2,xc+w/2,yc+h/2], outline='red', width=3)
    return np.array(img)

class DiceLoss(nn.Module):
    def __init__(self, eps=1e-6): super().__init__(); self.eps=eps
    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)
        tgt_oh = torch.zeros_like(probs).scatter_(1, targets.unsqueeze(1), 1)
        inter = (probs*tgt_oh).sum(dim=(2,3))
        card  = (probs+tgt_oh).sum(dim=(2,3))
        return 1 - ((2*inter+self.eps)/(card+self.eps)).mean()

print('Helpers ready.')

Using device: cuda
Helpers ready.


---
## Section 2.1 — Regularization effect of BatchNorm

Trains two VGG11 classifiers (with and without BN) for 11 epochs on a small subset.
Logs per-epoch train/val loss, activation distribution from the 3rd conv layer, and gradient norms.
This shows BN's effect on convergence speed and activation stability.

In [7]:
# VGG11 without BatchNorm (for comparison)
def _conv_relu(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=True),
        nn.ReLU(inplace=True),
    )

class VGG11_NoBN(nn.Module):
    def __init__(self, in_channels=3):
        super().__init__()
        self.block1 = nn.Sequential(_conv_relu(in_channels, 64));  self.pool1 = nn.MaxPool2d(2,2)
        self.block2 = nn.Sequential(_conv_relu(64, 128));           self.pool2 = nn.MaxPool2d(2,2)
        self.block3 = nn.Sequential(_conv_relu(128,256), _conv_relu(256,256)); self.pool3 = nn.MaxPool2d(2,2)
        self.block4 = nn.Sequential(_conv_relu(256,512), _conv_relu(512,512)); self.pool4 = nn.MaxPool2d(2,2)
        self.block5 = nn.Sequential(_conv_relu(512,512), _conv_relu(512,512)); self.pool5 = nn.MaxPool2d(2,2)
    def forward(self, x):
        x = self.pool1(self.block1(x)); x = self.pool2(self.block2(x))
        x = self.pool3(self.block3(x)); x = self.pool4(self.block4(x))
        return self.pool5(self.block5(x))

class VGG11Classifier_NoBN(nn.Module):
    def __init__(self, num_classes=37):
        super().__init__()
        self.encoder = VGG11_NoBN()
        self.pool = nn.AdaptiveAvgPool2d((7,7))
        self.flatten = nn.Flatten()
        self.head = nn.Sequential(
            nn.Linear(512*7*7, 4096), nn.ReLU(inplace=True),
            nn.Linear(4096, 4096),    nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes)
        )
    def forward(self, x):
        return self.head(self.flatten(self.pool(self.encoder(x))))

print('NoBN model defined.')

NoBN model defined.


In [9]:
# Small data loaders for the ablation (40 train batches, 15 val batches)
ds_train = OxfordIIITPetDataset('./data', 'trainval', 224)
ds_val   = OxfordIIITPetDataset('./data', 'test', 224)
train_loader_small = list(DataLoader(ds_train, batch_size=64, shuffle=True))
val_loader_small   = list(DataLoader(ds_val,   batch_size=64))

def get_activation_stats(model, loader, layer, num_batches=10):
    activations = []
    def hook_fn(m, i, o): activations.append(o.detach().cpu().flatten())
    handle = layer.register_forward_hook(hook_fn)
    model.eval()
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= num_batches: break
            model(batch['image'].to(device))
    handle.remove()
    return torch.cat(activations).numpy()

def run_bn_ablation(use_bn=True):
    run_name = 'BN_ON' if use_bn else 'BN_OFF'
    wandb.init(project='da6401-assignment2', name=run_name,
               tags=['2.1-bn-ablation'], reinit=True,
               config={'use_bn': use_bn, 'epochs': 11, 'lr': 1e-3})

    model = VGG11Classifier().to(device) if use_bn else VGG11Classifier_NoBN().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    if use_bn:
        hook_layer = model.encoder.block3[0][2]  # ReLU after BN
    else:
        hook_layer = model.encoder.block3[0][1]  # ReLU (no BN)

    for epoch in range(11):
        model.train()
        train_loss, total, grad_norm_sum, nbatches = 0.0, 0, 0.0, 0
        for batch in train_loader_small:
            imgs   = batch['image'].to(device)
            labels = batch['class_id'].to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            gn = nn.utils.clip_grad_norm_(model.parameters(), 5.0).item()
            optimizer.step()
            bs = imgs.size(0)
            train_loss += loss.item()*bs; total += bs
            grad_norm_sum += gn; nbatches += 1
        train_loss /= total

        model.eval()
        val_loss, vtotal = 0.0, 0
        with torch.no_grad():
            for batch in val_loader_small:
                imgs   = batch['image'].to(device)
                labels = batch['class_id'].to(device)
                loss   = criterion(model(imgs), labels)
                bs = imgs.size(0); val_loss += loss.item()*bs; vtotal += bs
        val_loss /= vtotal

        act = get_activation_stats(model, val_loader_small, hook_layer, num_batches=10)
        print(f'[{run_name}] Epoch {epoch:2d} | train={train_loss:.4f} val={val_loss:.4f} | act_mean={act.mean():.4f} act_std={act.std():.4f}')

        wandb.log({
            'train_loss':        train_loss,
            'val_loss':          val_loss,
            'activation_mean':   float(act.mean()),
            'activation_std':    float(act.std()),
            'activation_hist':   wandb.Histogram(act, num_bins=64),
            'grad_norm':         grad_norm_sum / nbatches,
        }, step=epoch)

    wandb.finish()

run_bn_ablation(True)   # BN_ON
run_bn_ablation(False)  # BN_OFF

[PetDataset] split=trainval | samples=3680
[PetDataset] split=test | samples=3669


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


[BN_ON] Epoch  0 | train=4.3955 val=10.7791 | act_mean=0.5445 act_std=0.7981
[BN_ON] Epoch  1 | train=4.0502 val=6.6989 | act_mean=0.4672 act_std=0.6836
[BN_ON] Epoch  2 | train=3.9193 val=4.3861 | act_mean=0.4062 act_std=0.5893
[BN_ON] Epoch  3 | train=3.6415 val=5.1934 | act_mean=0.4385 act_std=0.6629
[BN_ON] Epoch  4 | train=3.4171 val=5.1349 | act_mean=0.4369 act_std=0.6530
[BN_ON] Epoch  5 | train=3.1236 val=4.9128 | act_mean=0.4813 act_std=0.7185
[BN_ON] Epoch  6 | train=2.8947 val=5.0487 | act_mean=0.4633 act_std=0.6961
[BN_ON] Epoch  7 | train=2.6554 val=4.7468 | act_mean=0.4818 act_std=0.6940
[BN_ON] Epoch  8 | train=2.3742 val=4.5490 | act_mean=0.3734 act_std=0.5811
[BN_ON] Epoch  9 | train=2.0779 val=4.9420 | act_mean=0.3869 act_std=0.5962
[BN_ON] Epoch 10 | train=2.1389 val=5.7918 | act_mean=0.4104 act_std=0.5909


activation_mean,█▅▂▄▄▅▅▅▁▂▃
activation_std,█▄▁▄▃▅▅▅▁▁▁
grad_norm,█▅▄▄▃▃▃▂▂▂▁
train_loss,█▇▇▆▅▄▃▃▂▁▁
val_loss,█▄▁▂▂▂▂▁▁▂▃
activation_mean,0.41043
activation_std,0.5909
grad_norm,7.80991
train_loss,2.13886
val_loss,5.79178


[BN_OFF] Epoch  0 | train=3.7144 val=3.6109 | act_mean=0.0020 act_std=0.0131
[BN_OFF] Epoch  1 | train=3.6115 val=3.6109 | act_mean=0.0023 act_std=0.0128
[BN_OFF] Epoch  2 | train=3.6114 val=3.6109 | act_mean=0.0025 act_std=0.0146
[BN_OFF] Epoch  3 | train=3.6113 val=3.6109 | act_mean=0.0025 act_std=0.0146
[BN_OFF] Epoch  4 | train=3.6111 val=3.6109 | act_mean=0.0025 act_std=0.0146
[BN_OFF] Epoch  5 | train=3.6111 val=3.6109 | act_mean=0.0025 act_std=0.0146
[BN_OFF] Epoch  6 | train=3.6111 val=3.6109 | act_mean=0.0025 act_std=0.0146
[BN_OFF] Epoch  7 | train=3.6111 val=3.6109 | act_mean=0.0025 act_std=0.0146
[BN_OFF] Epoch  8 | train=3.6111 val=3.6109 | act_mean=0.0025 act_std=0.0146
[BN_OFF] Epoch  9 | train=3.6111 val=3.6109 | act_mean=0.0025 act_std=0.0146
[BN_OFF] Epoch 10 | train=3.6111 val=3.6109 | act_mean=0.0025 act_std=0.0146


activation_mean,▁▆█████████
activation_std,▂▁█████████
grad_norm,█▁▁▁▁▁▁▁▁▁▁
train_loss,█▁▁▁▁▁▁▁▁▁▁
val_loss,█▄▁▂▂▁▂▂▃▃▄
activation_mean,0.00246
activation_std,0.01464
grad_norm,0.12349
train_loss,3.61106
val_loss,3.61089


---
## Section 2.2 — Dropout comparison (no dropout / p=0.2 / p=0.5)

Trains the same VGG11 classifier under 3 dropout settings for 15 epochs.
Logs train/val loss curves — overlay them in W&B report to show the generalization gap.

In [ ]:
ds_train = OxfordIIITPetDataset('./data', 'trainval', 224)
ds_val   = OxfordIIITPetDataset('./data', 'test', 224)
train_loader_d = DataLoader(ds_train, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader_d   = DataLoader(ds_val,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

for p, run_name in [(0.0, 'dropout_p0'), (0.2, 'dropout_p02'), (0.5, 'dropout_p05')]:
    wandb.init(project='da6401-assignment2', name=run_name,
               tags=['2.2-dropout-ablation'], reinit=True,
               config={'dropout_p': p, 'epochs': 15})

    model     = VGG11Classifier(num_classes=37, dropout_p=p).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

    for epoch in range(1, 16):
        model.train()
        tl, tc, tt = 0.0, 0, 0
        for batch in train_loader_d:
            imgs, labels = batch['image'].to(device), batch['class_id'].to(device)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            bs = imgs.size(0); tl += loss.item()*bs; tc += (out.detach().argmax(1)==labels).sum().item(); tt += bs
        scheduler.step()
        # recompute cleanly
        model.eval()
        tl2, tc2, tt2 = 0.0, 0, 0
        with torch.no_grad():
            for batch in train_loader_d:
                imgs, labels = batch['image'].to(device), batch['class_id'].to(device)
                out = model(imgs); loss = criterion(out, labels)
                bs = imgs.size(0); tl2 += loss.item()*bs; tc2 += (out.argmax(1)==labels).sum().item(); tt2 += bs
        tl2 /= tt2; ta = tc2/tt2

        vl, vc, vt = 0.0, 0, 0
        with torch.no_grad():
            for batch in val_loader_d:
                imgs, labels = batch['image'].to(device), batch['class_id'].to(device)
                out = model(imgs); loss = criterion(out, labels)
                bs = imgs.size(0); vl += loss.item()*bs; vc += (out.argmax(1)==labels).sum().item(); vt += bs
        vl /= vt; va = vc/vt

        print(f'[{run_name}] Epoch {epoch:2d} | train_loss={tl2:.4f} acc={ta:.3f} | val_loss={vl:.4f} acc={va:.3f}')
        wandb.log({'train_loss': tl2, 'train_acc': ta, 'val_loss': vl, 'val_acc': va,
                   'generalization_gap': vl - tl2}, step=epoch)

    wandb.finish()
    print(f'Done: {run_name}')

[PetDataset] split=trainval | samples=3680
[PetDataset] split=test | samples=3669


generalization_gap,▁
train_acc,▁
train_loss,▁
val_acc,▁
val_loss,▁
generalization_gap,0.30359
train_acc,0.1394
train_loss,3.40321
val_acc,0.08749
val_loss,3.7068


[dropout_p0] Epoch  1 | train_loss=3.2852 acc=0.161 | val_loss=3.6002 acc=0.120
[dropout_p0] Epoch  2 | train_loss=2.7369 acc=0.269 | val_loss=3.3748 acc=0.155
[dropout_p0] Epoch  3 | train_loss=2.8804 acc=0.293 | val_loss=4.0536 acc=0.143
[dropout_p0] Epoch  4 | train_loss=1.5054 acc=0.572 | val_loss=3.2453 acc=0.187
[dropout_p0] Epoch  5 | train_loss=1.1305 acc=0.684 | val_loss=3.8045 acc=0.185
[dropout_p0] Epoch  6 | train_loss=0.5670 acc=0.839 | val_loss=4.3706 acc=0.204
[dropout_p0] Epoch  7 | train_loss=0.4400 acc=0.895 | val_loss=4.8245 acc=0.214
[dropout_p0] Epoch  8 | train_loss=0.1701 acc=0.949 | val_loss=4.6237 acc=0.226
[dropout_p0] Epoch  9 | train_loss=0.0317 acc=0.993 | val_loss=4.6282 acc=0.246
[dropout_p0] Epoch 10 | train_loss=0.0052 acc=0.999 | val_loss=4.5624 acc=0.253
[dropout_p0] Epoch 11 | train_loss=0.0014 acc=1.000 | val_loss=4.4460 acc=0.252
[dropout_p0] Epoch 12 | train_loss=0.0009 acc=1.000 | val_loss=4.4624 acc=0.254
[dropout_p0] Epoch 13 | train_loss=0.000

generalization_gap,▁▂▂▃▅▇█████████
train_acc,▁▂▂▄▅▇▇████████
train_loss,█▇▇▄▃▂▂▁▁▁▁▁▁▁▁
val_acc,▁▃▂▄▄▅▆▆▇██████
val_loss,▃▂▅▁▃▆█▇▇▇▆▆▆▇▆
generalization_gap,4.41502
train_acc,1
train_loss,0.00076
val_acc,0.25756
val_loss,4.41578


Done: dropout_p0


[dropout_p02] Epoch  1 | train_loss=3.7453 acc=0.141 | val_loss=4.1422 acc=0.102
[dropout_p02] Epoch  2 | train_loss=2.9211 acc=0.231 | val_loss=3.4545 acc=0.131
[dropout_p02] Epoch  3 | train_loss=3.2633 acc=0.296 | val_loss=4.0740 acc=0.153
[dropout_p02] Epoch  4 | train_loss=1.9937 acc=0.472 | val_loss=3.3375 acc=0.192
[dropout_p02] Epoch  5 | train_loss=1.9422 acc=0.498 | val_loss=4.1759 acc=0.171
[dropout_p02] Epoch  6 | train_loss=0.8024 acc=0.774 | val_loss=4.0513 acc=0.200
[dropout_p02] Epoch  7 | train_loss=0.5164 acc=0.854 | val_loss=4.5391 acc=0.225
[dropout_p02] Epoch  8 | train_loss=0.0862 acc=0.979 | val_loss=4.1766 acc=0.241
[dropout_p02] Epoch  9 | train_loss=0.0351 acc=0.994 | val_loss=4.2255 acc=0.244
[dropout_p02] Epoch 10 | train_loss=0.0069 acc=0.999 | val_loss=4.0702 acc=0.265
[dropout_p02] Epoch 11 | train_loss=0.0018 acc=1.000 | val_loss=3.9868 acc=0.278
[dropout_p02] Epoch 12 | train_loss=0.0011 acc=1.000 | val_loss=4.0194 acc=0.278
[dropout_p02] Epoch 13 | tra

generalization_gap,▁▁▂▃▄▆█████████
train_acc,▁▂▂▄▄▆▇████████
train_loss,█▆▇▅▅▂▂▁▁▁▁▁▁▁▁
val_acc,▁▂▃▄▄▅▆▆▆▇█████
val_loss,▆▂▅▁▆▅█▆▆▅▅▅▅▅▅
generalization_gap,3.98908
train_acc,1
train_loss,0.00077
val_acc,0.28427
val_loss,3.98985


Done: dropout_p02


[dropout_p05] Epoch  1 | train_loss=6.0428 acc=0.081 | val_loss=7.0677 acc=0.062
[dropout_p05] Epoch  2 | train_loss=3.1645 acc=0.209 | val_loss=3.8903 acc=0.126
[dropout_p05] Epoch  3 | train_loss=2.4501 acc=0.328 | val_loss=3.2560 acc=0.174
[dropout_p05] Epoch  4 | train_loss=2.1383 acc=0.440 | val_loss=3.1483 acc=0.197
[dropout_p05] Epoch  5 | train_loss=1.5997 acc=0.567 | val_loss=3.1455 acc=0.204
[dropout_p05] Epoch  6 | train_loss=1.0055 acc=0.699 | val_loss=3.4607 acc=0.217
[dropout_p05] Epoch  7 | train_loss=0.4658 acc=0.882 | val_loss=3.2968 acc=0.250
[dropout_p05] Epoch  8 | train_loss=0.2793 acc=0.930 | val_loss=3.5048 acc=0.236
[dropout_p05] Epoch  9 | train_loss=0.0941 acc=0.985 | val_loss=3.4659 acc=0.270
[dropout_p05] Epoch 10 | train_loss=0.0421 acc=0.993 | val_loss=3.4327 acc=0.286
[dropout_p05] Epoch 11 | train_loss=0.0119 acc=1.000 | val_loss=3.4475 acc=0.289
[dropout_p05] Epoch 12 | train_loss=0.0042 acc=1.000 | val_loss=3.4545 acc=0.301
[dropout_p05] Epoch 13 | tra

---
## Section 2.3 — Transfer learning strategies (frozen / partial / full)

Trains the UNet segmentation model under 3 encoder strategies for 15 epochs each.
Logs val Dice and loss — overlay in W&B report to compare convergence and final performance.

In [9]:
ds_train = OxfordIIITPetDataset('./data', 'trainval', 224)
ds_val   = OxfordIIITPetDataset('./data', 'test', 224)
train_loader_s = DataLoader(ds_train, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_loader_s   = DataLoader(ds_val,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

ce_loss   = nn.CrossEntropyLoss()
dice_loss = DiceLoss()

strategies = [
    ('frozen',  'Freeze all encoder — only decoder trains'),
    ('partial', 'Freeze blocks 1-3, unfreeze blocks 4-5'),
    ('full',    'Unfreeze entire network end-to-end'),
]

for strategy, description in strategies:
    wandb.init(project='da6401-assignment2', name=f'seg-{strategy}',
               tags=['2.3-transfer-learning'], reinit=True,
               config={'strategy': strategy, 'epochs': 15, 'description': description})

    model = VGG11UNet(
        num_classes=3,
        pretrained_encoder='checkpoints/classifier.pth',
        freeze_encoder=True,   # start fully frozen always
        dropout_p=0.5,
    ).to(device)

    # Unfreeze selectively
    if strategy == 'partial':
        for block in ['block4', 'block5']:
            for p in getattr(model.encoder, block).parameters():
                p.requires_grad = True
    elif strategy == 'full':
        for p in model.encoder.parameters():
            p.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'[{strategy}] Trainable params: {trainable:,} / {total:,}')
    wandb.config.update({'trainable_params': trainable})

    lr = 1e-4 if strategy == 'full' else 3e-4
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

    for epoch in range(1, 16):
        model.train()
        tl, td, tt = 0.0, 0.0, 0
        for batch in train_loader_s:
            imgs  = batch['image'].to(device)
            masks = batch['mask'].to(device)
            optimizer.zero_grad()
            logits = model(imgs)
            loss   = ce_loss(logits, masks) + dice_loss(logits, masks)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            bs = imgs.size(0); tl += loss.item()*bs; td += dice_score(logits.detach(), masks)*bs; tt += bs
        scheduler.step()
        tl /= tt; td /= tt

        model.eval()
        vl, vd, vpc, vpt = 0.0, 0.0, 0, 0
        with torch.no_grad():
            for batch in val_loader_s:
                imgs  = batch['image'].to(device)
                masks = batch['mask'].to(device)
                logits = model(imgs)
                loss   = ce_loss(logits, masks) + dice_loss(logits, masks)
                bs = imgs.size(0)
                vl  += loss.item()*bs
                vd  += dice_score(logits, masks)*bs
                vpc += (logits.argmax(1)==masks).sum().item()
                vpt += masks.numel()
        vl /= len(ds_val); vd /= len(ds_val); vpx = vpc/vpt

        print(f'[{strategy}] Epoch {epoch:2d} | train_loss={tl:.4f} dice={td:.3f} | val_loss={vl:.4f} dice={vd:.3f} pix={vpx:.3f}')
        wandb.log({'train_loss': tl, 'train_dice': td,
                   'val_loss': vl,   'val_dice': vd, 'val_pixel_acc': vpx}, step=epoch)

    wandb.finish()
    print(f'Done: {strategy}')

[PetDataset] split=trainval | samples=3680
[PetDataset] split=test | samples=3669


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


[UNet] Encoder loaded from checkpoints/classifier.pth
[frozen] Trainable params: 10,257,091 / 19,480,323
[frozen] Epoch  1 | train_loss=1.2525 dice=0.660 | val_loss=1.0295 dice=0.737 pix=0.773
[frozen] Epoch  2 | train_loss=0.9478 dice=0.749 | val_loss=0.8871 dice=0.771 pix=0.803
[frozen] Epoch  3 | train_loss=0.8559 dice=0.777 | val_loss=0.8254 dice=0.787 pix=0.816
[frozen] Epoch  4 | train_loss=0.7943 dice=0.799 | val_loss=0.8532 dice=0.785 pix=0.814
[frozen] Epoch  5 | train_loss=0.7634 dice=0.808 | val_loss=0.7769 dice=0.799 pix=0.827
[frozen] Epoch  6 | train_loss=0.7131 dice=0.827 | val_loss=0.7762 dice=0.799 pix=0.828
[frozen] Epoch  7 | train_loss=0.6790 dice=0.838 | val_loss=0.7431 dice=0.812 pix=0.836
[frozen] Epoch  8 | train_loss=0.6416 dice=0.851 | val_loss=0.7694 dice=0.803 pix=0.834
[frozen] Epoch  9 | train_loss=0.6080 dice=0.862 | val_loss=0.7541 dice=0.811 pix=0.839
[frozen] Epoch 10 | train_loss=0.5724 dice=0.874 | val_loss=0.7367 dice=0.816 pix=0.841
[frozen] Epoch 

train_dice,▁▄▄▅▅▆▆▆▇▇▇████
train_loss,█▅▄▄▄▃▃▃▂▂▂▁▁▁▁
val_dice,▁▄▅▅▆▆▇▆▇▇▇████
val_loss,█▅▃▄▂▂▁▂▂▁▁▁▁▁▁
val_pixel_acc,▁▄▅▅▆▆▇▇▇██████
train_dice,0.90488
train_loss,0.4735
val_dice,0.82318
val_loss,0.73128
val_pixel_acc,0.84451


Done: frozen


...

train_dice,▁▄▅▅▆▆▆▇▇▇█████
train_loss,█▅▄▄▄▃▃▃▂▂▂▁▁▁▁
val_dice,▂▁▅▆▅▇▇▇▇██████
val_loss,█▆▄▄▄▂▂▂▁▂▁▁▁▁▁
val_pixel_acc,▁▂▅▅▅▇▇▇▇▇█████
train_dice,0.92843
train_loss,0.38532
val_dice,0.87437
val_loss,0.56543
val_pixel_acc,0.88489


Done: partial


[UNet] Encoder loaded from checkpoints/classifier.pth
[full] Trainable params: 19,480,323 / 19,480,323
[full] Epoch  1 | train_loss=1.1403 dice=0.695 | val_loss=0.8296 dice=0.786 pix=0.823
[full] Epoch  2 | train_loss=0.7860 dice=0.816 | val_loss=0.6912 dice=0.833 pix=0.856
[full] Epoch  3 | train_loss=0.6882 dice=0.843 | val_loss=0.6449 dice=0.844 pix=0.863
[full] Epoch  4 | train_loss=0.6239 dice=0.861 | val_loss=0.6220 dice=0.850 pix=0.869
[full] Epoch  5 | train_loss=0.5774 dice=0.874 | val_loss=0.6092 dice=0.855 pix=0.868
[full] Epoch  6 | train_loss=0.5357 dice=0.887 | val_loss=0.5767 dice=0.868 pix=0.879
[full] Epoch  7 | train_loss=0.4946 dice=0.898 | val_loss=0.5572 dice=0.871 pix=0.885
[full] Epoch  8 | train_loss=0.4504 dice=0.912 | val_loss=0.5458 dice=0.876 pix=0.887
[full] Epoch  9 | train_loss=0.4198 dice=0.920 | val_loss=0.5239 dice=0.884 pix=0.893
[full] Epoch 10 | train_loss=0.3805 dice=0.931 | val_loss=0.5126 dice=0.888 pix=0.896
[full] Epoch 11 | train_loss=0.3552 d

train_dice,▁▄▅▆▆▆▇▇▇▇█████
train_loss,█▅▄▄▃▃▃▂▂▂▁▁▁▁▁
val_dice,▁▄▅▅▅▆▇▇▇██████
val_loss,█▅▄▄▃▃▂▂▁▁▁▁▁▁▁
val_pixel_acc,▁▄▅▅▅▆▇▇▇██████
train_dice,0.94963
train_loss,0.30111
val_dice,0.89303
val_loss,0.50883
val_pixel_acc,0.89925


Done: full


---
## Section 2.4 — Feature maps: first vs last conv layer

Passes a single dog image through the trained classifier and logs feature map grids
from block1 (first conv) and block5 (last conv before pooling).

In [8]:
import torchvision.utils as vutils

wandb.init(project='da6401-assignment2', name='feature-maps',
           tags=['2.4-feature-maps'], reinit='finish_previous')

model = VGG11Classifier(num_classes=37).to(device)
ckpt  = torch.load('checkpoints/classifier.pth', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

# Find a dog image (species == 2)
ds_val = OxfordIIITPetDataset('./data', 'test', 224)
dog_img = None
for i in range(len(ds_val)):
    s = ds_val[i]
    if s['species'].item() == 2:
        dog_img  = s['image'].unsqueeze(0).to(device)
        dog_orig = denorm(s['image'])
        break

assert dog_img is not None, 'No dog image found'

feat_block1, feat_block5 = [], []
h1 = model.encoder.block1[0][0].register_forward_hook(lambda m,i,o: feat_block1.append(o.detach().cpu()))
h5 = model.encoder.block5[0][0].register_forward_hook(lambda m,i,o: feat_block5.append(o.detach().cpu()))

with torch.no_grad():
    model(dog_img)
h1.remove(); h5.remove()

def make_grid(feat, max_ch=16):
    maps = feat[0][0][:max_ch].unsqueeze(1)      # [16, 1, H, W]
    # maps = (maps - maps.min()) / (maps.max() - maps.min() + 1e-5)
    maps_min = maps.view(maps.size(0), -1).min(dim=1)[0].view(-1,1,1,1)
    maps_max = maps.view(maps.size(0), -1).max(dim=1)[0].view(-1,1,1,1)
    maps = (maps - maps_min) / (maps_max - maps_min + 1e-5)
    grid = vutils.make_grid(maps, nrow=4, padding=2)  # [3, H', W'] — broadcast to RGB
    return (grid.permute(1, 2, 0).numpy() * 255).astype('uint8')

grid1 = make_grid(feat_block1)  # block1 conv: 64ch, 224x224 → shows edge detectors
grid5 = make_grid(feat_block5)  # block5 conv: 512ch, 14x14  → shows semantic parts

with torch.no_grad():
    pred_class = model(dog_img).argmax(1).item()

wandb.log({
    'input_image':        wandb.Image(dog_orig, caption='Input dog image'),
    'feature_map/block1': wandb.Image(grid1, caption='First layer: edge and texture detectors'),
    'feature_map/block5': wandb.Image(grid5, caption='Last layer: high-level semantic features'),
    'predicted_class_id': pred_class,
})

print(f'Predicted class: {pred_class}')
print('Feature maps logged.')
wandb.finish()

[PetDataset] split=test | samples=3669
Predicted class: 1
Feature maps logged.


predicted_class_id,▁
predicted_class_id,1


---
## Section 2.5 — Object detection table (val set)

Logs a W&B table with 15 val images showing GT boxes (green) and predicted boxes (red),
along with IoU score and whether classification was correct.
Uses the val set (NOT test) since test annotations are missing bboxes.

In [ ]:
from multitask import MultiTaskPerceptionModel

wandb.init(project='da6401-assignment2', name='detection-table',
           tags=['2.5-detection'], reinit=True)

mt_model = MultiTaskPerceptionModel().to(device)
mt_model.eval()

ds_val = OxfordIIITPetDataset('./data', 'trainval', 224)  # trainval has bbox annotations
val_loader_det = DataLoader(ds_val, batch_size=1, shuffle=True)

def iou_single(b1, b2, eps=1e-6):
    xc1,yc1,w1,h1 = b1; xc2,yc2,w2,h2 = b2
    ix = max(0, min(xc1+w1/2, xc2+w2/2) - max(xc1-w1/2, xc2-w2/2))
    iy = max(0, min(yc1+h1/2, yc2+h2/2) - max(yc1-h1/2, yc2-h2/2))
    inter = ix*iy; union = w1*h1 + w2*h2 - inter
    return float(inter / (union + eps))

table = wandb.Table(columns=['image', 'overlay', 'iou', 'conf', 'cls_correct', 'note'])
count = 0

with torch.no_grad():
    for batch in val_loader_det:
        if count >= 15: break
        imgs   = batch['image'].to(device)
        labels = batch['class_id'].to(device)
        boxes  = batch['bbox']

        out       = mt_model(imgs)
        pred_box  = out['localization'][0].cpu().numpy()
        pred_cls  = out['classification'][0].argmax().item()
        gt_box    = boxes[0].numpy()
        gt_cls    = labels[0].item()
        conf      = float(torch.softmax(out['classification'][0], dim=0).max())
        iou       = iou_single(gt_box, pred_box)
        correct   = pred_cls == gt_cls

        # Flag failure cases: high confidence but low IoU
        note = ''
        if conf > 0.5 and iou < 0.3:
            note = 'HIGH-CONF LOW-IOU failure'
        elif iou < 0.1:
            note = 'missed'

        overlay = draw_boxes(denorm(imgs[0]), gt_box, pred_box)

        table.add_data(
            wandb.Image(denorm(imgs[0])),
            wandb.Image(overlay, caption=f'Green=GT Red=Pred IoU={iou:.3f}'),
            round(iou, 4),
            round(conf, 4),
            correct,
            note,
        )
        count += 1

wandb.log({'detection_table': table})
print(f'Logged {count} samples to detection table.')
wandb.finish()

---
## Section 2.6 — Segmentation evaluation: Dice vs Pixel Accuracy

Logs 5 sample images (original / GT trimap / predicted trimap) and plots
both Dice and Pixel Accuracy on the val set to show why Dice is a better metric.

In [ ]:
wandb.init(project='da6401-assignment2', name='seg-evaluation',
           tags=['2.6-segmentation'], reinit=True)

unet = VGG11UNet(num_classes=3, pretrained_encoder='checkpoints/classifier.pth').to(device)
unet_ckpt = torch.load('checkpoints/unet.pth', map_location=device)
unet.load_state_dict(unet_ckpt['model_state_dict'])
unet.eval()

ds_val = OxfordIIITPetDataset('./data', 'test', 224)
val_loader_seg = DataLoader(ds_val, batch_size=16, shuffle=False, num_workers=2)

# --- Log 5 sample images ---
seg_table = wandb.Table(columns=['original', 'gt_mask', 'pred_mask', 'dice', 'pixel_acc'])
sample_loader = DataLoader(ds_val, batch_size=5, shuffle=True)
sample_batch  = next(iter(sample_loader))
sample_imgs   = sample_batch['image'].to(device)
sample_masks  = sample_batch['mask']

with torch.no_grad():
    preds = unet(sample_imgs).argmax(1).cpu().numpy()

for i in range(5):
    orig_np  = denorm(sample_imgs[i])
    gt_np    = TRIMAP_COLORS[sample_masks[i].numpy()]
    pred_np  = TRIMAP_COLORS[preds[i]]
    # per-image dice
    p = torch.tensor(preds[i]); m = sample_masks[i]
    fp = (p==0).float(); ft = (m==0).float()
    eps = 1e-6
    d  = float((2*(fp*ft).sum()+eps)/(fp.sum()+ft.sum()+eps))
    pa = float((p==m).float().mean())
    seg_table.add_data(
        wandb.Image(orig_np),
        wandb.Image(gt_np,   caption='GT: green=fg red=bg yellow=uncertain'),
        wandb.Image(pred_np, caption=f'Pred: Dice={d:.3f} PixAcc={pa:.3f}'),
        round(d, 4), round(pa, 4),
    )

wandb.log({'seg_samples': seg_table})

# --- Compute full val set metrics ---
total_dice, total_pxc, total_pxt, total_n = 0.0, 0, 0, 0
macro_dice_per_class = [0.0, 0.0, 0.0]  # fg, bg, uncertain

with torch.no_grad():
    for batch in val_loader_seg:
        imgs  = batch['image'].to(device)
        masks = batch['mask'].to(device)
        logits = unet(imgs)
        pred_m = logits.argmax(1)
        bs = imgs.size(0)

        # Macro Dice across all 3 classes
        probs = torch.softmax(logits, dim=1)
        for c in range(3):
            fp = (pred_m==c).float(); ft = (masks==c).float()
            macro_dice_per_class[c] += float((2*(fp*ft).sum()+eps)/(fp.sum()+ft.sum()+eps)) * bs

        total_dice += dice_score(logits, masks) * bs
        total_pxc  += (pred_m==masks).sum().item()
        total_pxt  += masks.numel()
        total_n    += bs

mean_dice    = total_dice / total_n
pixel_acc    = total_pxc  / total_pxt
macro_dice   = sum(macro_dice_per_class) / (3 * total_n)

wandb.log({
    'val/fg_dice':     macro_dice_per_class[0] / total_n,
    'val/bg_dice':     macro_dice_per_class[1] / total_n,
    'val/unc_dice':    macro_dice_per_class[2] / total_n,
    'val/macro_dice':  macro_dice,
    'val/pixel_acc':   pixel_acc,
})

print(f'Val Macro Dice: {macro_dice:.4f} | Pixel Acc: {pixel_acc:.4f}')
print('Note: Pixel acc is inflated by the dominant background class.')
wandb.finish()

---
## Section 2.7 — Wild images (3 internet pets)

Downloads 3 pet images from the internet (not from Oxford dataset) and runs the
full MultiTaskPerceptionModel pipeline on them, logging bbox + segmentation overlays.

In [ ]:
import requests
from io import BytesIO
from multitask import MultiTaskPerceptionModel

wandb.init(project='da6401-assignment2', name='wild-images',
           tags=['2.7-wild'], reinit=True)

mt_model = MultiTaskPerceptionModel().to(device)
mt_model.eval()

# --- Replace these with any 3 freely available pet image URLs ---
# These are example URLs — swap with ones you find on Google Images or Unsplash
wild_images = [
    ('shiba_inu',     'https://images.unsplash.com/photo-1544568100-847a948585b9'),
    ('persian_cat',   'https://images.unsplash.com/photo-1592194996308-7b43878e84a6'),
    ('golden_retrvr', 'https://images.unsplash.com/photo-1552053831-71594a27632d'),
]

wild_table = wandb.Table(columns=['name', 'original', 'bbox_pred', 'seg_overlay', 'pred_class_id'])

for name, url in wild_images:
    try:
        resp = requests.get(url, timeout=10)
        img_pil = Image.open(BytesIO(resp.content)).convert('RGB').resize((224, 224))
        img_np  = np.array(img_pil)
        t = torch.from_numpy(img_np).permute(2,0,1).float()/255.0
        t = ((t - MEAN)/STD).unsqueeze(0).to(device)

        with torch.no_grad():
            out = mt_model(t)

        pred_cls  = out['classification'][0].argmax().item()
        pred_box  = out['localization'][0].cpu().numpy()
        pred_mask = out['segmentation'][0].argmax(0).cpu().numpy()

        # Bbox overlay (red only — no GT for wild images)
        bbox_vis = img_pil.copy()
        d = ImageDraw.Draw(bbox_vis)
        xc,yc,w,h = pred_box
        d.rectangle([xc-w/2,yc-h/2,xc+w/2,yc+h/2], outline='red', width=3)

        # Segmentation overlay (semi-transparent)
        mask_rgb  = Image.fromarray(TRIMAP_COLORS[pred_mask])
        mask_rgba = mask_rgb.convert('RGBA')
        mask_rgba.putalpha(120)
        overlay   = img_pil.convert('RGBA')
        overlay   = Image.alpha_composite(overlay, mask_rgba).convert('RGB')

        wild_table.add_data(
            name,
            wandb.Image(img_np),
            wandb.Image(np.array(bbox_vis), caption=f'Pred bbox'),
            wandb.Image(np.array(overlay),  caption=f'Seg overlay'),
            pred_cls,
        )
        print(f'{name}: class={pred_cls} box={pred_box.round(1)}')
    except Exception as e:
        print(f'Failed to load {name}: {e}')

wandb.log({'wild_image_results': wild_table})
wandb.finish()

---
## Section 2.8 — Meta-analysis: final val metrics summary

Evaluates the full MultiTaskPerceptionModel on the val set and logs a
comprehensive summary of all three task metrics.

In [ ]:
from multitask import MultiTaskPerceptionModel

wandb.init(project='da6401-assignment2', name='meta-analysis',
           tags=['2.8-meta'], reinit=True)

mt_model = MultiTaskPerceptionModel().to(device)
mt_model.eval()

ds_val = OxfordIIITPetDataset('./data', 'trainval', 224)
val_loader_meta = DataLoader(ds_val, batch_size=32, shuffle=False, num_workers=2)

cls_correct = 0
iou_sum = 0.0
dice_sum = 0.0
pxc, pxt = 0, 0
total = 0
all_preds, all_labels = [], []
eps = 1e-6

with torch.no_grad():
    for batch in val_loader_meta:
        imgs   = batch['image'].to(device)
        labels = batch['class_id'].to(device)
        boxes  = batch['bbox'].to(device)
        masks  = batch['mask'].to(device)

        out = mt_model(imgs)
        bs  = imgs.size(0)

        # Classification
        pred_cls = out['classification'].argmax(1)
        cls_correct += (pred_cls == labels).sum().item()
        all_preds.extend(pred_cls.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

        # Localization
        pb = out['localization']
        pw, ph = pb[:,2].clamp(0), pb[:,3].clamp(0)
        px1, px2 = pb[:,0]-pw/2, pb[:,0]+pw/2
        py1, py2 = pb[:,1]-ph/2, pb[:,1]+ph/2
        tw, th   = boxes[:,2], boxes[:,3]
        tx1, tx2 = boxes[:,0]-tw/2, boxes[:,0]+tw/2
        ty1, ty2 = boxes[:,1]-th/2, boxes[:,1]+th/2
        iw = (torch.min(px2,tx2)-torch.max(px1,tx1)).clamp(0)
        ih = (torch.min(py2,ty2)-torch.max(py1,ty1)).clamp(0)
        inter = iw*ih; union = pw*ph + tw*th - inter
        iou_sum += (inter/(union+eps)).sum().item()

        # Segmentation
        pred_m = out['segmentation'].argmax(1)
        fp = (pred_m==0).float(); ft = (masks==0).float()
        dice_sum += ((2*(fp*ft).sum(1)+eps)/(fp.sum(1)+ft.sum(1)+eps)).sum().item()
        pxc += (pred_m==masks).sum().item()
        pxt += masks.numel()
        total += bs

cls_acc   = cls_correct / total
macro_f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
mean_iou  = iou_sum / total
mean_dice = dice_sum / total
pix_acc   = pxc / pxt

print(f'Classification Accuracy : {cls_acc:.4f}')
print(f'Macro F1                : {macro_f1:.4f}')
print(f'Mean IoU                : {mean_iou:.4f}')
print(f'Mean Dice               : {mean_dice:.4f}')
print(f'Pixel Accuracy          : {pix_acc:.4f}')

wandb.log({
    'final/cls_accuracy': cls_acc,
    'final/macro_f1':     macro_f1,
    'final/mean_iou':     mean_iou,
    'final/mean_dice':    mean_dice,
    'final/pixel_acc':    pix_acc,
})

# Bar chart summary
import matplotlib.pyplot as plt
metrics = {'Cls Acc': cls_acc, 'Macro F1': macro_f1, 'Mean IoU': mean_iou, 'Mean Dice': mean_dice, 'Pixel Acc': pix_acc}
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(metrics.keys(), metrics.values(), color='steelblue')
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title('Final val metrics — Multi-Task Pipeline')
for i, (k, v) in enumerate(metrics.items()):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=10)
plt.tight_layout()
wandb.log({'final/summary_chart': wandb.Image(fig)})
plt.close()

wandb.finish()
print('Meta-analysis done.')